In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("data/rent.csv")
TARGET_CATEGORIES = ["Appartements", "Maisons et Villas"]
REMOVE_TOP_3_PERCENT_OUTLIERS = True

df = pd.read_csv(DATA_PATH)
df = df[df["category"].isin(TARGET_CATEGORIES)].copy()

if REMOVE_TOP_3_PERCENT_OUTLIERS:
    p97 = df["price"].quantile(0.97)
    df = df[df["price"] <= p97].copy()

df = df.drop_duplicates().reset_index(drop=True)

feature_columns = ["room_count", "bathroom_count", "size", "city", "region", "category"]
X = df[feature_columns]
y = np.log1p(df["price"].values)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    )

categorical_features = ["city", "region", "category"]
numeric_features = ["room_count", "bathroom_count", "size"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        ),
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
    ]
)

model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

pred_log = model.predict(X_test)
pred = np.expm1(pred_log)
actual = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(actual, pred))
mae = mean_absolute_error(actual, pred)
r2 = r2_score(actual, pred)

print("Dataset used:", DATA_PATH)
print("Rows used:", len(df))
print("Top 3% outliers removed:", REMOVE_TOP_3_PERCENT_OUTLIERS)
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Dataset used: data/rent.csv
Rows used: 6412
Top 3% outliers removed: True
RMSE: 601.7189578906238
MAE: 394.39015673663846
R2: 0.381356950936087


In [2]:
from pathlib import Path
import joblib

MODEL_OUTPUT_PATH = Path("models/rent_model.joblib")
MODEL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to: {MODEL_OUTPUT_PATH.resolve()}")

Model saved to: /home/marwen/Desktop/WEB_PROJECT/aimicroservice/models-training/price_prediction_model/models/rent_model.joblib
